In [45]:
!pip install -q streamlit joblib scikit-learn pandas numpy

In [46]:
import pandas as pd
import numpy as np
import joblib

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

In [47]:
url = "https://raw.githubusercontent.com/royarka251/InternCircle-Data-Science-AI/main/Task-5-First-Machine-Learning-Classifier/Titanic.csv"

df = pd.read_csv(url)

df.head()

,survived,pclass,sex,age,sibsp,parch,fare,embarked,class,who,adult_male,deck,embark_town,alive,alone
0,0,3,male,22.0,1,0,7.2500,S,Third,man,True,NaN,Southampton,no,False
1,1,1,female,38.0,1,0,71.2833,C,First,woman,False,C,Cherbourg,yes,False
2,1,3,female,26.0,0,0,7.9250,S,Third,woman,False,NaN,Southampton,yes,True
3,1,1,female,35.0,1,0,53.1000,S,First,woman,False,C,Southampton,yes,False
4,0,3,male,35.0,0,0,8.0500,S,Third,man,True,NaN,Southampton,no,True


In [48]:
print("Dataset Shape:", df.shape)
print("\nColumns:")
print(df.columns.tolist())

print("\nMissing Values:")
print(df.isnull().sum())

Dataset Shape: (891, 15)

Columns:
['survived', 'pclass', 'sex', 'age', 'sibsp', 'parch', 'fare', 'embarked', 'class', 'who', 'adult_male', 'deck', 'embark_town', 'alive', 'alone']

Missing Values:
survived         0
pclass           0
sex              0
age            177
sibsp            0
parch            0
fare             0
embarked         2
class            0
who              0
adult_male       0
deck           688
embark_town      2
alive            0
alone            0
dtype: int64


In [49]:
features = [
    "pclass",
    "sex",
    "age",
    "sibsp",
    "parch",
    "fare",
    "embarked"
]

X = df[features]
y = df["survived"]

In [50]:
X = X.copy()

X["age"] = X["age"].fillna(X["age"].median())
X["fare"] = X["fare"].fillna(X["fare"].median())
X["embarked"] = X["embarked"].fillna(X["embarked"].mode()[0])

print(X.isnull().sum())

pclass      0
sex         0
age         0
sibsp       0
parch       0
fare        0
embarked    0
dtype: int64


In [51]:
categorical_features = ["sex", "embarked"]

numerical_features = [
    "pclass",
    "age",
    "sibsp",
    "parch",
    "fare"
]

In [52]:
preprocessor = ColumnTransformer(
    transformers=[
        (
            "categorical",
            OneHotEncoder(handle_unknown="ignore"),
            categorical_features
        ),
        (
            "numerical",
            "passthrough",
            numerical_features
        )
    ]
)

In [53]:
model = DecisionTreeClassifier(
    random_state=42,
    max_depth=5
)

In [54]:
pipeline = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        ("classifier", model)
    ]
)

In [55]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

In [56]:
pipeline.fit(X_train, y_train)

Pipeline(steps=[('preprocessor',
                 ColumnTransformer(transformers=[('categorical',
                                                  OneHotEncoder(handle_unknown='ignore'),
                                                  ['sex', 'embarked']),
                                                 ('numerical', 'passthrough',
                                                  ['pclass', 'age', 'sibsp',
                                                   'parch', 'fare'])])),
                ('classifier',
                 DecisionTreeClassifier(max_depth=5, random_state=42))])

In [57]:
y_pred = pipeline.predict(X_test)

accuracy = accuracy_score(y_test, y_pred)

print("Accuracy:", accuracy)

Accuracy: 0.7653631284916201


In [58]:
print(classification_report(y_test, y_pred))

              precision    recall  f1-score   support

           0       0.77      0.88      0.82       110
           1       0.75      0.58      0.66        69

    accuracy                           0.77       179
   macro avg       0.76      0.73      0.74       179
weighted avg       0.76      0.77      0.76       179



In [59]:
print(confusion_matrix(y_test, y_pred))

[[97 13]
 [29 40]]


In [60]:
joblib.dump(pipeline, "model.pkl")

['model.pkl']

In [61]:
import os

print(os.path.getsize("model.pkl"), "bytes")

7515 bytes


In [62]:
loaded_model = joblib.load("model.pkl")

sample = pd.DataFrame({
    "pclass": [1],
    "sex": ["female"],
    "age": [25],
    "sibsp": [0],
    "parch": [0],
    "fare": [80],
    "embarked": ["C"]
})

prediction = loaded_model.predict(sample)

print("Prediction:", prediction[0])

Prediction: 1


In [63]:
app_code = '''
import streamlit as st
import pandas as pd
import joblib

st.set_page_config(
    page_title="Titanic Survival Predictor",
    page_icon="🚢",
    layout="centered"
)

st.title("🚢 Titanic Survival Predictor")
st.write("Enter passenger details to predict survival.")

@st.cache_resource
def load_model():
    return joblib.load("model.pkl")

model = load_model()

st.sidebar.header("Passenger Information")

pclass = st.sidebar.selectbox(
    "Passenger Class",
    [1, 2, 3]
)

sex = st.sidebar.selectbox(
    "Sex",
    ["male", "female"]
)

age = st.sidebar.slider(
    "Age",
    min_value=0.0,
    max_value=80.0,
    value=30.0,
    step=1.0
)

sibsp = st.sidebar.slider(
    "Number of Siblings/Spouses",
    min_value=0,
    max_value=8,
    value=0
)

parch = st.sidebar.slider(
    "Number of Parents/Children",
    min_value=0,
    max_value=6,
    value=0
)

fare = st.sidebar.slider(
    "Fare",
    min_value=0.0,
    max_value=600.0,
    value=30.0,
    step=1.0
)

embarked = st.sidebar.selectbox(
    "Port of Embarkation",
    ["S", "C", "Q"]
)

input_data = pd.DataFrame({
    "pclass": [pclass],
    "sex": [sex],
    "age": [age],
    "sibsp": [sibsp],
    "parch": [parch],
    "fare": [fare],
    "embarked": [embarked]
})

st.subheader("Passenger Details")
st.dataframe(input_data, use_container_width=True)

if st.button("Predict Survival"):
    prediction = model.predict(input_data)[0]

    if prediction == 1:
        st.success("🎉 Prediction: Passenger is likely to survive.")
    else:
        st.error("Prediction: Passenger is unlikely to survive.")

    if hasattr(model, "predict_proba"):
        probability = model.predict_proba(input_data)[0]

        st.write(
            f"Survival probability: {probability[1] * 100:.2f}%"
        )
'''

with open("app.py", "w", encoding="utf-8") as f:
    f.write(app_code)

print("app.py created successfully.")

app.py created successfully.


In [64]:
requirements = """streamlit
pandas
numpy
scikit-learn
joblib
"""

with open("requirements.txt", "w") as f:
    f.write(requirements)

print("requirements.txt created successfully.")

requirements.txt created successfully.


In [65]:
readme = """# Task 6 - Streamlit Interactive ML Web App

## Objective

Deploy a trained machine learning classifier as an interactive web application using Streamlit.

## Model

Decision Tree Classifier

## Dataset

Titanic Dataset

## Features

- Passenger Class
- Sex
- Age
- Number of Siblings/Spouses
- Number of Parents/Children
- Fare
- Embarkation Port

## Technologies

- Python
- Pandas
- NumPy
- Scikit-learn
- Joblib
- Streamlit

## How It Works

The trained machine learning pipeline is loaded using Joblib.
Users enter passenger information through interactive controls.
The application generates a real-time survival prediction.

## Files

- app.py
- model.pkl
- requirements.txt
- Task_6_Streamlit_Interactive_ML_Web_App.ipynb
"""

with open("README.md", "w") as f:
    f.write(readme)

print("README.md created successfully.")

README.md created successfully.


In [66]:
import os

print("Files in current folder:")
for file in os.listdir():
    print(file)

Files in current folder:
.config
app.py
model.pkl
requirements.txt
README.md
sample_data


In [67]:
import joblib

model = joblib.load("model.pkl")

print("Model loaded successfully!")
print(type(model))

Model loaded successfully!
<class 'sklearn.pipeline.Pipeline'>


In [68]:
%%writefile app.py

import streamlit as st
import pandas as pd
import joblib

# Load trained model
model = joblib.load("model.pkl")

st.set_page_config(
    page_title="Titanic Survival Predictor",
    page_icon="🚢",
    layout="centered"
)

st.title("🚢 Titanic Survival Prediction")
st.write("Enter passenger information to predict survival.")

# Inputs
pclass = st.selectbox(
    "Passenger Class",
    [1, 2, 3]
)

sex = st.selectbox(
    "Sex",
    ["male", "female"]
)

age = st.slider(
    "Age",
    min_value=0.0,
    max_value=80.0,
    value=30.0
)

sibsp = st.number_input(
    "Number of Siblings/Spouses",
    min_value=0,
    max_value=8,
    value=0
)

parch = st.number_input(
    "Number of Parents/Children",
    min_value=0,
    max_value=6,
    value=0
)

fare = st.number_input(
    "Fare",
    min_value=0.0,
    value=30.0
)

embarked = st.selectbox(
    "Embarkation Port",
    ["S", "C", "Q"]
)

if st.button("Predict Survival"):

    input_data = pd.DataFrame({
        "pclass": [pclass],
        "sex": [sex],
        "age": [age],
        "sibsp": [sibsp],
        "parch": [parch],
        "fare": [fare],
        "embarked": [embarked]
    })

    prediction = model.predict(input_data)

    if prediction[0] == 1:
        st.success("🚢 Prediction: Passenger is likely to survive.")
    else:
        st.error("🚢 Prediction: Passenger is unlikely to survive.")

Overwriting app.py


In [69]:
import joblib

model = joblib.load("model.pkl")

print(model)

Pipeline(steps=[('preprocessor',
                 ColumnTransformer(transformers=[('categorical',
                                                  OneHotEncoder(handle_unknown='ignore'),
                                                  ['sex', 'embarked']),
                                                 ('numerical', 'passthrough',
                                                  ['pclass', 'age', 'sibsp',
                                                   'parch', 'fare'])])),
                ('classifier',
                 DecisionTreeClassifier(max_depth=5, random_state=42))])


In [70]:
%%writefile requirements.txt

streamlit
pandas
numpy
scikit-learn
joblib

Overwriting requirements.txt


# Task 6 - Streamlit Interactive ML Web App

## Project Overview

This project deploys a Titanic survival prediction
machine learning model using Streamlit.

## Features

- Interactive passenger inputs
- Machine learning prediction
- Real-time prediction output
- Trained model loaded using Joblib
- Streamlit web interface

## Technologies

- Python
- Pandas
- NumPy
- Scikit-learn
- Joblib
- Streamlit

## Dataset

Titanic Dataset

## Model

Machine Learning Classification Model

## Files

- app.py
- model.pkl
- requirements.txt
- README.md
- Task_6_Streamlit_Interactive_ML_Web_App.ipynb

## Live Demo

Coming soon